# 💳 Smart Outreach Optimization with Multi-Armed Bandits
### Thompson Sampling vs UCB vs Epsilon-Greedy — Bank Marketing Dataset

## 📌 Overview
A Portuguese bank wants to know **which outreach strategy** (season of contact,
number of calls, previous campaign result) leads to the highest chance a
customer opens a term deposit.

Instead of running a slow, wasteful traditional A/B test — where bad strategies
keep getting tested long after we know they're bad — this project uses
**Multi-Armed Bandit algorithms** to balance exploration and exploitation,
converging faster on the best strategy while minimizing lost conversions
("regret") along the way.

Since this is **historical (logged) data** rather than a live interactive
environment, we use the **Replay Method** — a standard offline bandit
evaluation technique — to fairly simulate how each algorithm would have
performed if it had been making decisions in real time.

## 🎯 What's Compared
| Algorithm | Type |
|---|---|
| Thompson Sampling | Bayesian |
| UCB (Upper Confidence Bound) | Frequentist |
| Epsilon-Greedy | Heuristic |
| Random Selection | Baseline (≈ traditional A/B testing) |

## 📦 Step 1: Setup & Configuration

In [17]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import beta

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

THEME = {
    "bg": "#0d0221",
    "primary": "#7c3aed",
    "light": "#c4b5fd",
    "accent": "#f472b6",
    "text": "#c4b5fd",
}

def apply_dark_theme(ax, title, xlabel="", ylabel=""):
    """Apply consistent dark purple theme to a matplotlib axis."""
    ax.set_facecolor(THEME["bg"])
    ax.set_title(title, fontsize=15, color="white", pad=15)
    ax.set_xlabel(xlabel, color=THEME["light"])
    ax.set_ylabel(ylabel, color=THEME["light"])
    ax.tick_params(colors=THEME["light"])
    ax.grid(True, alpha=0.2, color=THEME["primary"])

plt.rcParams["figure.facecolor"] = THEME["bg"]

## 📂 Step 2: Load Data

In [18]:
dataset = pd.read_csv("bank-full.csv", sep=";")
print(f"Shape : {dataset.shape}")
dataset.head()

Shape : (45211, 17)


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


## 🎯 Step 3: Arm Engineering

Each **arm** represents a distinct outreach strategy, defined by three signals
that a bank could realistically act on:

- **Season** — derived from contact month
- **Campaign Frequency** — how many times this customer was contacted this campaign
- **Previous Outcome** — result of the prior campaign for this customer

Combining these gives us actionable strategies like:
*"Contact in Autumn, on the 1st attempt, if the previous campaign failed."*

In [19]:
SEASON_MAP = {
    "dec": "Winter", "jan": "Winter", "feb": "Winter",
    "mar": "Spring", "apr": "Spring", "may": "Spring",
    "jun": "Summer", "jul": "Summer", "aug": "Summer",
    "sep": "Autumn", "oct": "Autumn", "nov": "Autumn",
}

def bucket_campaign(n_contact: int) -> str:
    """Bucket number of contacts into Low / Medium / High frequency."""
    if n_contact == 1:
        return "Low(1)"
    elif n_contact <= 3:
        return "Medium(2-3)"
    return "High(4+)"

dataset["season"] = dataset["month"].map(SEASON_MAP)
dataset["camp_bucket"] = dataset["campaign"].apply(bucket_campaign)
dataset["arm"] = dataset["season"] + " | " + dataset["camp_bucket"] + " | " + dataset["poutcome"]
dataset["reward"] = (dataset["y"] == "yes").astype(int)

print(f"Total arms: {dataset['arm'].nunique()}")
dataset[["season", "camp_bucket", "poutcome", "arm", "reward"]].head()

Total arms: 48


,season,camp_bucket,poutcome,arm,reward
0,Spring,Low(1),unknown,Spring | Low(1) | unknown,0
1,Spring,Low(1),unknown,Spring | Low(1) | unknown,0
2,Spring,Low(1),unknown,Spring | Low(1) | unknown,0
3,Spring,Low(1),unknown,Spring | Low(1) | unknown,0
4,Spring,Low(1),unknown,Spring | Low(1) | unknown,0


## 🧹 Step 4: Filter Sparse Arms

Arms with very few logged samples make the Replay Method unreliable — there
simply isn't enough historical data to fairly estimate how an algorithm would
perform if it picked that arm. We drop arms below a minimum sample threshold.

In [20]:
MIN_ARM_SAMPLES = 30

arm_count = dataset["arm"].value_counts()
valid_arm = arm_count[arm_count >= MIN_ARM_SAMPLES].index
dataset = dataset[dataset["arm"].isin(valid_arm).reset_index(drop=True)]

print(f"Arms kept: {len(valid_arm)} / {len(arm_count)}")
print(f"Rows kept: {len(dataset)} / 45211")

Arms kept: 44 / 48
Rows kept: 45110 / 45211


## 🤖 Step 5: Bandit Algorithms

We implement four strategies for selecting an arm at each round:

- **Thompson Sampling** — Bayesian approach. Each arm has a Beta(α, β)
  distribution representing our belief about its success rate. At each round,
  we sample from each arm's distribution and pick the highest sample —
  naturally balancing exploration and exploitation.
- **UCB (Upper Confidence Bound)** — Frequentist approach. Picks the arm with
  the highest "optimistic" estimate (mean reward + a confidence bonus that
  shrinks as an arm gets tried more).
- **Epsilon-Greedy** — Simple heuristic. With probability ε, explore randomly;
  otherwise, exploit the current best-known arm.
- **Random** — Baseline, roughly equivalent to traditional A/B testing where
  every strategy gets equal traffic regardless of performance.

In [21]:
class ThompsonSampling:
    """Bayesian bandit using Beta-Bernoulli conjugate priors."""

    def __init__(self, n_arms: int):
        self.alpha = np.ones(n_arms)  # successes + 1
        self.beta = np.ones(n_arms)    # failures + 1

    def select_arm(self) -> int:
        samples = np.random.beta(self.alpha, self.beta)
        return int(np.argmax(samples))

    def update(self, arm: int, reward: int):
        self.alpha[arm] += reward
        self.beta[arm] += (1 - reward)

class UCB:
    """Upper Confidence Bound (UCB1)."""

    def __init__(self, n_arms: int):
        self.n_arms = n_arms
        self.counts = np.zeros(n_arms)
        self.values = np.zeros(n_arms)  # running mean reward per arm
        self.total_count = 0

    def select_arm(self) -> int:
        # Try every arm at least once before computing bounds
        if 0 in self.counts:
            return int(np.argmin(self.counts))

        confidence_bonus = np.sqrt(2 * np.log(self.total_count) / self.counts)
        ucb_values = self.values + confidence_bonus
        return int(np.argmax(ucb_values))

    def update(self, arm: int, reward: int):
        self.total_count += 1
        self.counts[arm] += 1
        n = self.counts[arm]
        # incremental mean update
        self.values[arm] += (reward - self.values[arm]) / n


class EpsilonGreedy:
    """Epsilon-Greedy with a fixed exploration rate."""

    def __init__(self, n_arms: int, epsilon: float = 0.1):
        self.epsilon = epsilon
        self.counts = np.zeros(n_arms)
        self.values = np.zeros(n_arms)

    def select_arm(self) -> int:
        if np.random.random() < self.epsilon:
            return np.random.randint(len(self.values))
        return int(np.argmax(self.values))

    def update(self, arm: int, reward: int):
        self.counts[arm] += 1
        n = self.counts[arm]
        self.values[arm] += (reward - self.values[arm]) / n


class RandomPolicy:
    """Baseline — picks an arm uniformly at random, ~ traditional A/B testing."""

    def __init__(self, n_arms: int):
        self.n_arms = n_arms

    def select_arm(self) -> int:
        return np.random.randint(self.n_arms)

    def update(self, arm: int, reward: int):
        pass  # no learning

## 🔁 Step 6: Replay Method (Offline Bandit Evaluation)

Since our data is historical (logged), we can't let an algorithm interact with
real customers. The **Replay Method** solves this:

1. Shuffle the logged data (removes any time-based bias).
2. Go through it row by row. At each row, ask the algorithm: *"which arm would
   you pick right now?"*
3. **Only if** the algorithm's chosen arm matches the arm that was actually
   used on that historical customer, we "accept" the event: feed the real
   logged reward back to the algorithm and count it as a round.
4. If it doesn't match, skip the row — we have no ground truth for what would
   have happened with a different arm, so we simply move on.

This gives an **unbiased estimate** of how the algorithm would have performed
online, using only historical data. It's a widely used technique in
industry (e.g. Yahoo! Research's "Contextual Bandit" paper) precisely for
this offline-evaluation scenario.

In [22]:
def reply_evaluation(dataset: pd.DataFrame, policy, arm_to_idx: dict, n_rounds: int = None):
    """
    Run the Replay Method to evaluate a bandit policy on logged data.

    Parameters
    ----------
    df : shuffled dataframe with 'arm' and 'reward' columns
    policy : an object with select_arm() and update(arm, reward)
    arm_to_idx : mapping from arm name -> integer index
    n_rounds : stop after this many accepted rounds (None = use all matches)

    Returns
    -------
    rewards : list of rewards received at each accepted round
    chosen_arms : list of arm indices chosen at each accepted round
    """

    rewards = []
    chosen_arms = []

    for _, row in dataset.iterrows():
        true_arm_idx = arm_to_idx[row["arm"]]
        chosen_arm = policy.select_arm()

        if chosen_arm == true_arm_idx:
            reward = row["reward"]
            policy.update(chosen_arm, reward)
            rewards.append(reward)
            chosen_arms.append(chosen_arm)

            if n_rounds is not None and len(rewards) >= n_rounds:
                break

    return rewards, chosen_arms

## ⚙️ Step 7: Running the Replay Simulation

We now run all four policies through the Replay Method on the same shuffled
data, using an equal number of accepted rounds so comparisons are fair.

In [23]:
# Shuffle once, reuse the same shuffled order for every policy
dataset_shuffled = dataset.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

arms = sorted(dataset["arm"].unique())
arm_to_idx = {arm: i for i, arm in enumerate(arms)}
n_arms = len(arms)
print(f"n_arms = {n_arms}")

N_ROUNDS = 2000   # target accepted rounds per policy

policies = {
    "Thompson Sampling": ThompsonSampling(n_arms),
    "UCB": UCB(n_arms),
    "Epsilon-Greedy": EpsilonGreedy(n_arms, epsilon=0.1),
    "Random": RandomPolicy(n_arms),
}

results = {}

for name, policy in policies.items():
    rewards, chosen_arms = reply_evaluation(dataset_shuffled, policy, arm_to_idx, n_rounds=N_ROUNDS)
    results[name] = {"reward": rewards, "chosen_arms": chosen_arms}
    print(f"{name}: {len(rewards)} accepted rounds, "
          f"avg reward = {np.mean(rewards):.4f}")

n_arms = 44
Thompson Sampling: 167 accepted rounds, avg reward = 0.2814
UCB: 164 accepted rounds, avg reward = 0.3476
Epsilon-Greedy: 290 accepted rounds, avg reward = 0.4862
Random: 1008 accepted rounds, avg reward = 0.1270


## 🔁 Step 7 (Revised): Multi-Pass Replay Method

With 44 arms, a single pass through ~45K logged rows isn't enough for
converged policies (like Thompson Sampling or UCB) to accumulate enough
matched rounds — they quickly focus on a few arms, so exact matches become
rare.

To fix this, we let each policy make **multiple passes** over freshly
reshuffled data until it reaches the target number of accepted rounds. This
is a standard adaptation of the Replay Method for limited historical data —
each pass uses a new random shuffle so we're not just repeating the exact
same sequence.

In [24]:
def reply_evaluation(dataset: pd.DataFrame, policy, arm_to_idx: dict,
                     n_rounds: int, max_passes: int = 50, random_state: int = None):
    """
    Run the Replay Method to evaluate a bandit policy on logged data,
    reshuffling and making multiple passes if needed to reach n_rounds.

    Parameters
    ----------
    df : dataframe with 'arm' and 'reward' columns (unshuffled is fine —
         we shuffle internally on every pass)
    policy : an object with select_arm() and update(arm, reward)
    arm_to_idx : mapping from arm name -> integer index
    n_rounds : target number of accepted rounds
    max_passes : safety cap on how many reshuffled passes to attempt
    random_state : base seed; each pass uses a different derived seed

    Returns
    -------
    rewards : list of rewards received at each accepted round
    chosen_arms : list of arm indices chosen at each accepted round
    """

    rewards = []
    chosen_arms = []

    for pass_num in range(max_passes):
        seed = None if random_state is None else random_state + pass_num
        dataset_pass = dataset.sample(frac=1, random_state=seed).reset_index(drop=True)

        for _, row in dataset_pass.iterrows():
            true_arm_idx = arm_to_idx[row["arm"]]
            chosen_arm = policy.select_arm()

            if chosen_arm == true_arm_idx:
                reward = row["reward"]
                policy.update(chosen_arm, reward)
                rewards.append(reward)
                chosen_arms.append(chosen_arm)

                if len(rewards) >= n_rounds:
                    return rewards, chosen_arms

    print(f"⚠️ Warning: only reached {len(rewards)}/{n_rounds} rounds after {max_passes} passes")
    return rewards, chosen_arms

## ⚙️ Step 8: Re-running with Multi-Pass Replay

In [25]:
N_ROUNDS = 2000

policies = {
    "Thompson Sampling": ThompsonSampling(n_arms),
    "UCB": UCB(n_arms),
    "Epsilon-Greedy": EpsilonGreedy(n_arms, epsilon=0.1),
    "Random": RandomPolicy(n_arms),
}

results = {}
for name, policy in policies.items():
    rewards, chosen_arms = reply_evaluation(
        dataset, policy, arm_to_idx, n_rounds=N_ROUNDS, random_state=RANDOM_STATE
    )
    results[name] = {"rewards": rewards, "chosen_arms": chosen_arms}
    print(f"{name}: {len(rewards)} accepted rounds, "
          f"avg reward = {np.mean(rewards):.4f}")

Thompson Sampling: 2000 accepted rounds, avg reward = 0.6230
UCB: 2000 accepted rounds, avg reward = 0.4495
Epsilon-Greedy: 2000 accepted rounds, avg reward = 0.4395
Random: 2000 accepted rounds, avg reward = 0.1160
